# TP2 Run All - Ronda_12 MaxStack

One-click run for the all-techniques stack: best historical prompts, token-balanced prompts, prompt order search, ablations, crossover, ensemble ranking, target-specific region/heuristic scoring, auxiliary multi-seed validation, and final fixed-seed renders.


## 1. Locate Project

In [ ]:
from pathlib import Path
import os
import sys

START = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [START, *START.parents] if (p / "src" / "search_multiseed_validate.py").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root containing src/search_multiseed_validate.py")
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)


## 2. Generate Ronda_12 Prompt Bank

In [ ]:
import subprocess

cmd = [
    sys.executable,
    str(PROJECT_ROOT / "src" / "gerar_ronda_12_maxstack.py"),
    "--output",
    str(PROJECT_ROOT / "prompts" / "refinement_ronda_12_maxstack.json"),
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


## 3. Token Audit

In [ ]:
import json

prompt_path = PROJECT_ROOT / "prompts" / "refinement_ronda_12_maxstack.json"
data = json.loads(prompt_path.read_text(encoding="utf-8"))
try:
    from transformers import CLIPTokenizer
    tok = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14", local_files_only=True)
    counts = []
    for target, entries in data.items():
        vals = [len(tok(entry["prompt"], truncation=False, add_special_tokens=True)["input_ids"]) for entry in entries]
        counts.extend(vals)
        print(target, "count", len(vals), "max", max(vals), "avg", round(sum(vals) / len(vals), 1), "over77", sum(v > 77 for v in vals))
    print("total", len(counts), "max", max(counts), "over77", sum(v > 77 for v in counts))
except Exception as exc:
    print("Token audit skipped:", exc)
    print({target: len(entries) for target, entries in data.items()})


## 4. MaxStack Multi-Seed Search

In [ ]:
import subprocess

RUN_IDENTITY = "notebook_ronda_12_maxstack"
VALIDATION_TOP_N = 34
ENSEMBLE_PER_METRIC = 9
FINAL_TOP_K = 12
SEED_OFFSETS = ["1", "2", "3"]

cmd = [
    sys.executable,
    str(PROJECT_ROOT / "src" / "search_multiseed_validate.py"),
    "--prompts", str(PROJECT_ROOT / "prompts" / "refinement_ronda_12_maxstack.json"),
    "--targets", str(PROJECT_ROOT / "TP2-students" / "students" / "tp2-chosen"),
    "--output-dir", str(PROJECT_ROOT / "TP2-students" / "students" / "outputs"),
    "--identity", RUN_IDENTITY,
    "--top-k", str(FINAL_TOP_K),
    "--stage1-save-k", "30",
    "--validation-top-n", str(VALIDATION_TOP_N),
    "--ensemble-per-metric", str(ENSEMBLE_PER_METRIC),
    "--seed-offsets", *SEED_OFFSETS,
    "--only", "1159_7.png", "9338.png",
    "--offline",
    "--disable-progress-bar",
    "--maxstack-scoring",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


## 5. Show Latest Results

In [ ]:
from IPython.display import display
from PIL import Image

outputs = PROJECT_ROOT / "TP2-students" / "students" / "outputs"
runs = sorted(outputs.glob("*notebook_ronda_12_maxstack"), key=lambda p: p.stat().st_mtime, reverse=True)
if not runs:
    raise FileNotFoundError("No Ronda_12 output folder found")
run_dir = runs[0]
print("Latest Ronda_12 run:", run_dir)
for path in sorted(run_dir.glob("top*_robust_fixed_seed.csv")):
    print("Final ranking:", path)
for path in sorted(run_dir.glob("contact_sheet_top*_robust.jpg")):
    print("Contact sheet:", path)
    display(Image.open(path))
